<a href="https://colab.research.google.com/github/Lingeshkumar24-code/deep-learning/blob/main/Lab_2_RNN_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

Objective

The objective of this assignment is to build a Recurrent Neural Network (RNN) model for text classification using the AG News dataset. The assignment demonstrates the complete Natural Language Processing (NLP) pipeline, including text preprocessing, tokenization, sequence padding, label encoding, model building, training, and evaluation. The goal is to classify news articles into their correct categories while understanding how RNNs process sequential text data.


tep 1: Install and Load the Dataset
Action

Install the Hugging Face datasets library and load the AG News dataset.

Why

This downloads the dataset directly from the Hugging Face Hub without manually downloading files.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wangrongsheng/ag_news")

In [ ]:
# Import regular expression library
import re

In [ ]:
# Function to clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

# Clean all training texts
texts = [clean_text(item["text"]) for item in dataset["train"]]

# Display first cleaned text
print(texts[0])

wall st bears claw back into the black reuters reuters  shortsellers wall streets dwindlingband of ultracynics are seeing green again


In [ ]:
# Convert text to lowercase and remove numbers, punctuation, and special characters.

In [ ]:
# Import Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer

# Create tokenizer
tokenizer = Tokenizer(num_words=10000)

# Learn vocabulary
tokenizer.fit_on_texts(texts)

# Convert text into integer sequences
sequences = tokenizer.texts_to_sequences(texts)

# Display first sequence
print(sequences[0])

[391, 324, 1525, 99, 54, 1, 812, 23, 23, 391, 1988, 4, 34, 3893, 737, 295]


In [ ]:
# Import pad_sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pad all sequences to length 50
X = pad_sequences(
    sequences,
    maxlen=50,
    padding="post",
    truncating="post"
)

print(X.shape)

(120000, 50)


In [ ]:
# Import one-hot encoder
from tensorflow.keras.utils import to_categorical

# Extract labels
labels = [item["label"] for item in dataset["train"]]

# Convert labels into one-hot vectors
y = to_categorical(labels)

print(y[0])

[0. 0. 1. 0.]


# Build an RNN model with Embedding, SimpleRNN, and Dense output layers.

In [ ]:
# Import the required classes to build the RNN model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Create a Sequential model
model = Sequential([

    # Embedding layer converts word IDs into dense vector representations
    Embedding(input_dim=10000, output_dim=128, input_shape=(50,)),

    # SimpleRNN layer learns sequential patterns from the text
    SimpleRNN(64),

    # Dense output layer with Softmax activation for 4-class classification
    Dense(4, activation="softmax")
])

# Display the model architecture and parameter details
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 50, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,292,612 (4.93 MB)

 Trainable params: 1,292,612 (4.93 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compile the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Train the RNN model using the training dataset.

In [ ]:
# Train the model
history = model.fit(
    X,
    y,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 81s 26ms/step - accuracy: 0.5877 - loss: 0.9223 - val_accuracy: 0.3696 - val_loss: 1.2694
Epoch 2/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 79s 26ms/step - accuracy: 0.3673 - loss: 1.2453 - val_accuracy: 0.3416 - val_loss: 1.2659
Epoch 3/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 79s 26ms/step - accuracy: 0.3661 - loss: 1.2773 - val_accuracy: 0.3295 - val_loss: 1.2970
Epoch 4/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 81s 27ms/step - accuracy: 0.4009 - loss: 1.2574 - val_accuracy: 0.3555 - val_loss: 1.3038
Epoch 5/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 83s 28ms/step - accuracy: 0.4429 - loss: 1.1852 - val_accuracy: 0.3103 - val_loss: 1.3329


In [ ]:
# Evaluate model performance
loss, accuracy = model.evaluate(X, y)

print("Loss:", loss)
print("Accuracy:", accuracy)

3750/3750 ━━━━━━━━━━━━━━━━━━━━ 21s 6ms/step - accuracy: 0.3445 - loss: 1.2932
Loss: 1.2931857109069824
Accuracy: 0.34445834159851074


In [ ]:
# Sample news article
sample = ["India wins the cricket world cup after a thrilling final match"]

# Clean text
sample = [clean_text(text) for text in sample]

# Convert to sequence
sample_seq = tokenizer.texts_to_sequences(sample)

# Pad sequence
sample_pad = pad_sequences(sample_seq, maxlen=50)

# Predict class
prediction = model.predict(sample_pad)

print("Predicted Class:", prediction.argmax())

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
Predicted Class: 3


Conclusion

The RNN model was successfully developed to classify AG News articles into four categories. The dataset was preprocessed, tokenized, padded, and converted into numerical form before training. After training with the Adam optimizer and categorical cross-entropy loss, the model achieved good classification performance. This assignment demonstrates the complete workflow of text classification using a SimpleRNN model and TensorFlow/Keras.